# Trees and Ensembles, Notebook 5
## Variable importance: what it means for a ratio to "matter", in each model and compared with the betas

**Prepared by:** David Díaz, with assistance from Claude (Anthropic) · **Environment:** Google Colab

### What this is about

After training any model, the question that comes from the business side is always the same:
**which variables carry weight?** A manager wants to know whether what drives the risk is liquidity or
leverage; a regulator wants to know that the model does not rely on something improper. Each family
of models answers in a different way, and in this notebook we put the answers side by side,
computed with the same data and the same partition as in the previous notebooks:

| Model | What it delivers | Has a sign | Assumes linearity |
|---|---|---|---|
| Linear or logistic regression | coefficients $\beta_j$ | yes | yes |
| Tree (ID3, CART) | **impurity**-based importance: how much impurity each variable reduced | no | no |
| Random forest, boosting | the same, averaged or accumulated over the trees | no | no |
| Any model | **permutation** importance: how much worse it gets if the variable is shuffled | no | no |
| Any model | per-observation contributions (**SHAP**) | yes, per observation | no |

### What you will learn today

1. What a regression coefficient measures and why you have to standardize to compare.
2. How a tree's impurity-based importance is computed, node by node, by hand and with `scikit-learn`.
3. How it is averaged in a forest, and a known bias of that measure.
4. Permutation importance, which works for any model.
5. What AdaBoost and gradient boosting report.
6. SHAP: recovering the sign, firm by firm.
7. How to read the differences between the measures without concluding that one of them is wrong.


In [ ]:

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings

import statsmodels.api as sm
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, roc_auc_score

warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)
print("Ready.")


## 1. The data, once more

The same 219 firms and the same partition (146 training, 73 test).


| Ratio | What it measures |
|---|---|
| `deuda_activos` | total debt / total assets (leverage) |
| `razon_corriente` | current assets / current liabilities (liquidity) |
| `ventas_deuda` | sales / total debt (ability to service the debt) |
| `ln_activos` | logarithm of total assets (size) |
| `roa` | net income / total assets (profitability) |
| `impago` | **what we want to predict:** 1 if the firm defaulted, 0 if not |


In [ ]:

# The 219 firms with financial ratios (the course default table, with 5 of its ratios).
# It is pasted right here so the notebook does not depend on any external file.
from io import StringIO
IMPAGO_CSV = """deuda_activos,razon_corriente,ventas_deuda,ln_activos,roa,impago
0.374,2.337,1.436,14.697,0.073,0.0
0.435,2.638,1.385,14.877,0.042,0.0
0.443,2.099,0.452,14.931,0.016,0.0
0.23,2.506,4.217,15.586,-0.041,0.0
0.317,2.273,3.116,15.708,0.021,0.0
0.312,3.282,4.842,15.785,0.047,0.0
0.628,1.386,2.91,14.066,0.064,0.0
0.64,1.777,2.895,14.236,0.068,0.0
0.719,1.44,1.505,14.697,0.075,0.0
0.551,1.078,1.944,15.564,0.113,0.0
0.541,0.991,2.091,15.581,0.022,0.0
0.575,0.995,1.787,15.738,0.035,0.0
0.454,3.266,7.321,14.877,0.064,0.0
0.574,3.017,3.159,14.457,0.07,0.0
0.526,3.333,1.962,14.399,0.077,0.0
0.724,0.8,1.625,14.036,0.016,0.0
0.508,1.406,4.356,14.349,0.406,0.0
0.494,1.051,2.299,14.515,0.117,0.0
0.572,1.557,2.106,14.016,0.126,0.0
0.578,1.498,2.033,14.293,0.086,0.0
0.558,1.605,2.035,14.52,0.099,0.0
0.314,2.127,10.633,13.738,0.087,0.0
0.467,1.661,5.599,14.219,0.094,0.0
0.499,1.482,3.109,14.424,0.067,0.0
0.259,3.734,5.692,14.172,0.234,0.0
0.202,4.804,7.312,14.417,0.195,0.0
0.268,3.663,6.08,14.897,0.228,0.0
0.409,1.671,4.523,14.253,0.12,0.0
0.307,2.171,6.817,14.38,0.172,0.0
0.258,3.481,10.557,14.454,0.274,0.0
0.393,1.55,3.81,14.68,0.024,0.0
0.558,1.102,2.597,14.108,0.005,0.0
0.432,1.365,3.534,14.01,0.119,0.0
0.53,1.281,3.099,12.687,0.272,0.0
0.244,3.411,16.602,13.143,0.489,0.0
0.354,2.877,3.112,13.311,0.075,0.0
0.381,1.56,2.749,14.696,0.074,0.0
0.411,2.198,2.46,14.839,0.043,0.0
0.401,1.954,2.067,14.915,0.037,0.0
0.387,1.955,2.616,14.835,0.342,0.0
0.438,1.029,2.619,14.929,0.023,0.0
0.413,1.44,2.869,14.889,-0.011,0.0
0.891,1.118,2.023,13.415,0.036,0.0
1.052,0.741,1.8,14.355,-0.093,0.0
0.466,0.899,1.509,14.364,0.034,0.0
0.491,1.196,2.335,14.414,-0.017,0.0
0.505,1.053,2.678,14.631,0.076,0.0
0.354,2.45,3.08,14.453,0.084,0.0
0.464,1.843,2.521,14.559,0.119,0.0
0.566,1.286,1.675,14.838,0.111,0.0
0.543,1.766,2.619,14.933,0.029,0.0
0.855,1.131,1.805,15.005,0.029,0.0
0.558,1.744,2.516,15.074,0.028,0.0
0.736,1.153,1.564,14.292,0.054,0.0
0.708,1.192,1.352,14.432,0.034,0.0
0.657,1.269,1.395,14.387,0.027,0.0
0.653,1.061,1.588,14.303,0.04,0.0
0.619,1.005,2.113,14.235,0.038,0.0
0.197,9.632,5.15,14.586,0.032,0.0
0.284,1.461,6.585,13.479,0.11,0.0
0.3,1.438,5.251,13.541,0.012,0.0
0.283,2.928,5.778,13.561,0.016,0.0
0.476,1.788,5.329,13.143,0.123,0.0
0.47,1.759,5.033,13.329,0.13,0.0
0.459,2.104,4.49,13.372,0.076,0.0
0.388,2.409,5.319,13.283,0.201,0.0
0.547,1.979,2.09,13.806,0.15,0.0
0.568,1.812,1.937,13.914,0.071,0.0
0.461,1.923,1.499,14.059,-0.013,0.0
0.538,2.174,1.801,14.208,0.025,0.0
0.497,1.878,1.917,14.284,0.075,0.0
0.208,3.306,5.181,15.336,-0.003,0.0
0.241,3.005,4.709,15.428,0.011,0.0
0.222,3.11,3.776,15.427,0.021,0.0
0.392,2.268,6.865,12.39,0.334,0.0
0.252,5.229,7.751,12.318,0.176,0.0
0.849,1.56,1.326,12.304,-0.186,0.0
0.36,2.671,5.196,12.142,0.186,0.0
0.028,34.514,71.252,12.295,0.292,0.0
0.631,1.369,1.835,13.284,0.136,0.0
0.507,1.464,1.625,13.732,-0.217,0.0
0.578,1.577,2.553,13.963,0.02,0.0
0.448,1.681,0.566,13.768,0.024,0.0
0.363,2.721,2.762,13.301,0.446,0.0
0.403,2.199,2.673,13.506,0.067,0.0
0.4,2.237,1.139,13.576,0.043,0.0
0.432,1.579,2.39,14.7,0.066,0.0
0.464,1.763,2.067,14.843,0.059,0.0
0.475,1.707,1.833,14.95,0.06,0.0
0.089,8.849,16.344,15.39,0.013,0.0
0.125,7.175,11.171,15.513,0.059,0.0
0.125,7.156,12.795,15.55,0.041,0.0
0.539,1.764,3.733,15.384,0.088,0.0
0.502,1.79,3.797,15.397,0.035,0.0
0.522,1.572,2.624,15.391,0.029,0.0
0.77,0.383,1.984,15.834,-0.046,0.0
0.746,0.465,3.31,16.041,0.057,0.0
0.666,0.539,3.919,16.051,0.018,0.0
0.14,8.346,5.975,15.361,0.008,0.0
0.183,5.927,4.699,15.443,0.026,0.0
0.24,4.46,3.828,15.549,0.016,0.0
0.749,1.39,2.739,15.184,0.015,0.0
0.72,1.212,2.976,15.14,0.006,0.0
0.824,0.937,3.12,15.016,-0.089,0.0
0.127,0.463,3.018,13.706,0.036,0.0
0.584,3.254,1.629,14.581,0.053,0.0
0.565,0.66,1.621,14.753,0.085,0.0
0.853,3.41,0.45,13.939,0.012,0.0
0.836,4.713,0.337,14.072,0.007,0.0
0.968,2.708,0.249,14.346,-0.016,0.0
0.387,2.077,1.735,15.354,-0.039,0.0
0.398,2.395,2.933,15.532,0.059,0.0
0.327,2.428,1.813,15.518,0.036,0.0
0.635,1.999,0.061,15.292,-0.005,0.0
0.279,4.784,0.637,14.664,0.016,0.0
0.162,2.572,1.5,14.58,0.057,0.0
0.21,1.923,15.689,12.954,0.112,0.0
0.434,0.844,4.415,12.837,-0.246,0.0
0.271,1.323,5.378,12.849,0.17,0.0
0.416,2.122,5.392,13.508,0.098,0.0
0.409,2.225,4.438,13.477,0.018,0.0
0.262,3.185,5.914,13.392,0.128,0.0
0.23,4.172,5.796,14.956,0.218,0.0
0.085,11.333,15.115,15.14,0.246,0.0
0.134,6.66,9.398,15.292,0.101,0.0
0.759,2.52,2.074,13.84,0.084,0.0
0.77,1.92,1.572,14.162,0.031,0.0
0.675,2.107,1.621,13.957,0.036,0.0
0.601,2.387,1.541,13.442,-0.214,0.0
0.733,1.376,1.151,13.28,-0.11,0.0
0.822,1.461,1.286,13.478,-0.026,0.0
0.15,9.243,7.907,13.92,0.068,0.0
0.103,13.87,11.417,14.067,0.135,0.0
0.054,35.477,17.746,14.134,0.107,0.0
0.672,1.436,1.373,13.419,0.145,1.0
0.292,0.205,1.514,13.547,0.171,1.0
0.614,1.341,1.041,13.703,0.069,1.0
0.507,1.49,3.369,13.853,0.203,1.0
0.7,1.17,0.148,14.649,0.092,1.0
0.616,2.062,1.508,14.571,0.074,1.0
0.622,1.136,0.098,14.945,0.043,1.0
0.731,1.134,0.852,15.429,0.041,1.0
0.747,0.97,0.859,15.636,0.037,1.0
0.485,1.984,3.443,13.503,0.059,1.0
0.599,1.421,2.111,13.788,0.033,1.0
0.685,1.229,1.801,13.965,0.009,1.0
0.675,0.869,1.083,13.874,0.096,1.0
0.828,0.878,0.992,14.654,0.018,1.0
0.696,1.079,3.177,14.413,0.072,1.0
1.314,0.703,0.86,12.145,0.072,1.0
1.458,0.588,0.01,11.974,-0.086,1.0
1.732,0.454,0.0,11.772,-0.171,1.0
0.743,0.921,1.933,13.546,0.185,1.0
0.752,1.488,2.063,13.671,0.202,1.0
1.074,1.367,1.418,13.605,0.13,1.0
0.297,0.154,0.94,16.088,0.064,1.0
0.268,1.354,0.861,16.111,0.025,1.0
0.09,1.484,2.472,16.149,0.019,1.0
0.405,3.01,2.249,15.232,0.022,1.0
0.488,2.13,2.073,15.416,0.03,1.0
0.528,1.87,2.078,15.514,0.071,1.0
0.715,1.106,1.3,15.805,0.055,1.0
0.66,1.287,1.822,15.719,0.018,1.0
0.678,1.252,1.754,15.89,0.029,1.0
0.631,1.578,1.041,14.786,0.02,1.0
0.635,1.733,1.055,14.82,0.026,1.0
0.659,1.585,1.171,14.771,0.028,1.0
0.689,1.368,0.95,12.36,-0.033,1.0
0.464,1.238,1.069,12.662,-0.162,1.0
0.845,1.356,0.552,12.717,-0.352,1.0
0.548,1.741,0.697,16.077,0.002,1.0
0.595,1.234,0.758,16.058,0.0,1.0
0.768,0.919,0.355,16.536,-0.08,1.0
0.535,3.214,3.409,12.725,0.186,1.0
0.588,2.784,4.476,12.623,0.303,1.0
0.683,2.317,4.213,12.453,0.327,1.0
0.42,1.553,1.755,15.717,0.012,1.0
0.397,2.599,2.095,15.712,0.028,1.0
0.348,2.223,1.268,15.703,0.021,1.0
0.592,1.518,2.17,14.555,0.047,1.0
0.608,1.401,1.509,14.678,0.014,1.0
0.661,0.001,0.941,14.767,-0.013,1.0
1.48,0.225,0.001,12.488,-0.078,1.0
1.575,0.324,0.044,12.488,-0.009,1.0
0.638,0.87,1.662,15.163,-0.005,1.0
0.624,0.906,1.764,15.229,0.03,1.0
0.631,1.037,1.033,8.626,0.086,1.0
0.27,2.783,10.977,13.105,0.19,1.0
0.471,2.03,6.381,13.449,0.224,1.0
0.292,3.383,9.848,13.567,0.22,1.0
0.701,1.253,3.098,15.845,0.056,1.0
0.799,0.047,2.534,15.942,0.048,1.0
0.772,1.24,2.471,16.078,0.054,1.0
1.181,0.804,0.944,12.801,0.075,1.0
0.464,1.293,2.927,12.743,0.092,1.0
0.514,1.105,2.764,12.839,0.082,1.0
0.386,2.301,6.078,12.668,0.399,1.0
0.723,1.327,1.854,13.612,0.125,1.0
0.695,1.38,1.813,13.589,0.085,1.0
0.638,1.322,4.07,13.505,0.286,1.0
0.555,1.614,3.204,13.949,0.13,1.0
0.594,1.528,2.718,14.176,0.125,1.0
1.401,0.709,0.208,12.076,-0.146,1.0
0.916,0.616,1.27,12.629,0.316,1.0
0.75,0.489,2.46,12.725,0.29,1.0
0.445,9.164,0.0,11.167,-0.737,1.0
0.767,0.664,1.149,12.778,-0.779,1.0
1.314,0.703,0.86,12.145,0.072,1.0
1.458,0.588,0.01,11.974,-0.086,1.0
1.732,0.454,0.0,11.772,-0.171,1.0
0.775,1.025,3.294,13.079,0.112,1.0
0.731,0.932,4.179,12.889,0.126,1.0
0.842,0.673,1.911,13.274,-0.04,1.0
0.733,1.226,2.588,13.42,0.091,1.0
0.707,1.325,2.385,13.461,0.06,1.0
1.209,0.656,1.554,13.272,-0.565,1.0
0.507,1.49,3.369,13.853,0.203,1.0
0.7,1.17,1.479,14.649,0.092,1.0
0.616,2.062,2.277,14.571,0.074,1.0"""
impago = pd.read_csv(StringIO(IMPAGO_CSV))
print("Firms:", len(impago), "  Fraction in default:", round(impago["impago"].mean(), 3))
impago.describe().round(3).T[["mean", "min", "50%", "max"]]


In [ ]:

# The same partition in every notebook of the set: 146 firms for training, 73 for testing
RATIOS = ["deuda_activos", "razon_corriente", "ventas_deuda", "ln_activos", "roa"]
rng = np.random.default_rng(2)
orden = rng.permutation(len(impago))
es_prueba = np.zeros(len(impago), dtype=bool); es_prueba[orden[:73]] = True
impago["conjunto"] = np.where(es_prueba, "prueba", "entrenamiento")

X = impago[RATIOS]; y = impago["impago"]
X_train, y_train = X[~es_prueba], y[~es_prueba]
X_test, y_test = X[es_prueba], y[es_prueba]
print("Training:", len(X_train), "  Test:", len(X_test))
print("Fraction in default: training", round(y_train.mean(), 3), " test", round(y_test.mean(), 3))


## 2. The classic answer: the betas of a regression

For a yes-or-no response, logistic regression models the logarithm of the odds
(the *log-odds*):

$$
\ln \frac{p}{1 - p} = \beta_0 + \beta_1 x_1 + \dots + \beta_5 x_5
$$

where $p$ is the probability of default, $x_j$ are the ratios and $\beta_j$ their coefficients. A
positive $\beta_j$ says the ratio raises the risk; a negative one, that it lowers it. Each coefficient is
read **holding the other ratios fixed**.

A coefficient has units: the one for the current ratio is "per unit of current ratio" and
the one for `ln_activos` "per unit of logarithm", so their sizes cannot be compared. To
compare them the variables are **standardized** (the mean is subtracted and they are divided by the
standard deviation, both computed with the training data only) and then each $\beta$ is read as the
effect of moving one standard deviation.

The coefficients are estimated by maximum likelihood, which is exactly minimizing the cross-entropy
of Notebook 1. `statsmodels` also delivers the standard error of each coefficient, and
with it a $z$ value and a $p$ value that say whether the coefficient is distinguishable from zero.


In [ ]:

mu, sd = X_train.mean(), X_train.std(ddof=0)
Z_train = (X_train - mu) / sd
Z_test = (X_test - mu) / sd

logit = sm.Logit(y_train, sm.add_constant(Z_train)).fit(disp=0)
tabla_betas = pd.DataFrame({"beta estandarizado": logit.params, "error estándar": logit.bse, "z": logit.tvalues, "p value": logit.pvalues}).round(3)
tabla_betas["lectura"] = np.where(logit.params > 0, "higher, more risk", "higher, less risk")
print(tabla_betas)
print()
p_test = logit.predict(sm.add_constant(Z_test))
print(f"Test accuracy (threshold 0.5): {accuracy_score(y_test, (p_test > 0.5).astype(int)):.3f}   AUC: {roc_auc_score(y_test, p_test):.3f}")


In [ ]:

betas = tabla_betas.drop("const").sort_values("beta estandarizado", key=abs)
fig = px.bar(betas, x="beta estandarizado", y=betas.index, orientation="h", error_x="error estándar",
             color=(betas["beta estandarizado"] > 0).map({True: "raises the risk", False: "lowers the risk"}),
             color_discrete_map={"raises the risk": "#B4562A", "lowers the risk": "#3B8A55"},
             title="Standardized betas of the logistic regression (with their standard error)")
fig.update_layout(yaxis_title="", legend_title="", height=380)
fig.show()


**How to read the table.** According to the regression, the ratio that moves the risk the most per standard
deviation is the current ratio, with a negative sign: more liquidity, less default. Notice two
things. First, with 146 firms the standard errors are large and several coefficients are not
distinguishable from zero ($p$ value greater than 0.10). Second, the sign of `roa` is positive: more
profitability associated with more risk makes no economic sense. It is the kind of result that
shows up when the ratios are correlated with each other (below, the correlation matrix:
`roa` correlates 0.40 with `ventas_deuda` and -0.36 with `deuda_activos`) and the relationship is not
linear: the regression splits the joint effect as best it can. In section 6 we will ask
a model without that assumption whether it agrees.

**What the coefficient promises and what it assumes.** It promises a clean reading: direction, size
and uncertainty. In exchange it assumes that the effect of each ratio is the same over its whole range
(linear in the log-odds) and that it does not depend on the other ratios (additive). When the relationship is
U-shaped or step-shaped (the one in Notebook 1), or when "collateral matters only if the debt is
high" (Notebook 2), the coefficient averages all of that into a single number.


In [ ]:

# How much the ratios correlate with each other (part of the explanation for the sign of roa)
print(X_train.corr().round(2))


## 3. In a tree: how much impurity each variable reduces

A tree has no coefficients, but at each node the chosen question **reduces the impurity** of the
group by a known amount: the information gain in ID3, the drop in Gini in CART, the
drop in variance in regression. The importance of a variable is the sum of those reductions
over all the nodes where it was asked about, weighting each node by the fraction of observations
that passed through it, and normalized so that the importances sum to 1:

$$
\text{imp}(x_j) \;\propto\; \sum_{t \,:\, x_j \text{ splits at } t} \frac{n_t}{n}
\left[ I(t) - \frac{n_{t,\text{left}}}{n_t} I(t_{\text{left}}) - \frac{n_{t,\text{right}}}{n_t} I(t_{\text{right}}) \right]
$$

where $n_t$ is how many observations reach node $t$, $n$ the total, and $I$ the impurity the
tree uses. The bracket is the impurity reduction that node achieves; the factor $n_t / n$
makes a split at the root, through which all observations pass, weigh more than one further down.

We compute it by hand by walking the internal structure of the `scikit-learn` tree
(`arbol.tree_`) and compare it with `feature_importances_`, which is exactly this calculation.


In [ ]:

arbol2 = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_train, y_train)
t = arbol2.tree_
n = t.n_node_samples[0]

def importancia_a_mano(t, nombres):
    acum = pd.Series(0.0, index=nombres)
    for nodo in range(t.node_count):
        izq, der = t.children_left[nodo], t.children_right[nodo]
        if izq == -1:            # leaf
            continue
        reduccion = t.impurity[nodo] - (t.n_node_samples[izq] / t.n_node_samples[nodo]) * t.impurity[izq] \
                                     - (t.n_node_samples[der] / t.n_node_samples[nodo]) * t.impurity[der]
        acum[nombres[t.feature[nodo]]] += t.n_node_samples[nodo] / t.n_node_samples[0] * reduccion
        print(f"node {nodo}: {nombres[t.feature[nodo]]:16s} n = {t.n_node_samples[nodo]:3d}   impurity {t.impurity[nodo]:.3f}   reduction {reduccion:.4f}   weighted {t.n_node_samples[nodo] / t.n_node_samples[0] * reduccion:.4f}")
    return acum / acum.sum()

imp_mano = importancia_a_mano(t, RATIOS)
print()
comparacion = pd.DataFrame({"by hand": imp_mano, "feature_importances_": arbol2.feature_importances_}).round(3)
print(comparacion)


The two columns match: `feature_importances_` is this sum. `ventas_deuda` takes the
largest share because it is the root and its split is seen by all 146 firms; `ln_activos` and `roa` are left at
zero because they were never asked about, even though they may have come a close second at some
node. Compare with the regression: there `ventas_deuda` was not first. It is not that one of the two is
wrong; they measure different things.

**The same for ID3, with entropy.** On the 14 firms of Notebook 2 the tree has three
internal nodes: debt at the root (14 firms), collateral in the high-debt branch (5) and credit history
in the medium-debt branch (5).


In [ ]:

# The 14 firms that applied for credit (the small course table)
credito = pd.DataFrame({
    "tamano":    ["pequena", "grande", "mediana", "pequena", "grande", "mediana", "pequena", "grande", "mediana", "pequena", "grande", "mediana", "pequena", "grande"],
    "deuda":     ["baja", "baja", "baja", "baja", "alta", "alta", "alta", "alta", "alta", "media", "media", "media", "media", "media"],
    "garantia":  ["no", "si", "no", "si", "si", "si", "no", "no", "no", "no", "si", "si", "no", "no"],
    "historial": ["bueno", "malo", "malo", "bueno", "bueno", "malo", "bueno", "malo", "bueno", "bueno", "bueno", "malo", "malo", "malo"],
    "paga":      ["si", "si", "si", "si", "si", "si", "no", "no", "no", "si", "si", "no", "no", "no"],
})
credito.index = range(1, 15)
credito


In [ ]:

def H(serie):
    p = serie.value_counts(normalize=True)
    return float(-(p * np.log2(p)).sum())

def ganancia(df, atributo):
    return H(df["paga"]) - sum(len(g) / len(df) * H(g["paga"]) for _, g in df.groupby(atributo))

nodos_id3 = [("root", credito, "deuda"), ("deuda = alta", credito[credito.deuda == "alta"], "garantia"), ("deuda = media", credito[credito.deuda == "media"], "historial")]
imp_id3 = pd.Series(0.0, index=["tamano", "deuda", "garantia", "historial"])
for nombre, grupo, atributo in nodos_id3:
    g = ganancia(grupo, atributo)
    imp_id3[atributo] += len(grupo) / len(credito) * g
    print(f"{nombre:14s} asks {atributo:10s} n = {len(grupo):2d}   gain {g:.3f}   weighted {len(grupo) / len(credito) * g:.3f}")
print()
print((imp_id3 / imp_id3.sum()).round(3))


Notice that collateral and history beat debt, which is the root: their questions leave pure
groups (gain 0.971 each) and, weighted by 5/14, give 0.347 against the root's 0.292. In such a
small tree the root does not always win. Size was never asked about: zero importance, even though at the
root it had a gain of 0.006.

**What this importance says and what it does not.** It says how much the tree used each variable to sort
the data, with relationships of any shape and interactions included. It does not say the direction of the
effect (the same variable can raise the risk in one branch and lower it in another), it has no units
and no standard error, and it inherits the tree's instability. And it has a known bias that we see in the
next section.


## 4. In a forest: averaging over the trees, and a bias worth knowing about

With many trees the calculation is the same in each one and then averaged. The average fixes the
instability: a variable that was left at zero in one tree shows up in others.


In [ ]:

bosque = RandomForestClassifier(n_estimators=500, random_state=0).fit(X_train, y_train)
imp_bosque = pd.Series(bosque.feature_importances_, index=RATIOS)
por_arbol = pd.DataFrame([a.feature_importances_ for a in bosque.estimators_], columns=RATIOS)
resumen = pd.DataFrame({"depth-2 tree": arbol2.feature_importances_, "random forest (average of 500)": imp_bosque,
                        "std. dev. across trees": por_arbol.std()}).round(3)
print(resumen)
print(f"\nForest test accuracy: {accuracy_score(y_test, bosque.predict(X_test)):.3f}")


**The bias toward variables with many values.** A numeric variable with many distinct
values offers many more candidate splits than a two-valued one, and so it tends to gain
importance "for free": it is the same problem as the identifier trap of Notebook 1. To
see it, we add two columns of pure noise, with no relationship to default: one with 146
distinct values and another with only two.


In [ ]:

rng = np.random.default_rng(5)
X_ruido = X_train.copy(); Xt_ruido = X_test.copy()
X_ruido["ruido_continuo"] = rng.normal(size=len(X_train)); Xt_ruido["ruido_continuo"] = rng.normal(size=len(X_test))
X_ruido["ruido_binario"] = rng.integers(0, 2, len(X_train)); Xt_ruido["ruido_binario"] = rng.integers(0, 2, len(X_test))

bosque_ruido = RandomForestClassifier(n_estimators=500, random_state=0).fit(X_ruido, y_train)
imp_ruido = pd.Series(bosque_ruido.feature_importances_, index=X_ruido.columns).round(3)
print(imp_ruido.sort_values(ascending=False))


The continuous noise, which carries no information at all, receives an impurity-based importance
comparable to that of real ratios; the binary noise, almost nothing. It is not that the forest "believes" the
continuous noise is useful: it is that, with 146 distinct values, there is always some split that sorts the
training data a little by chance, and the impurity counts it.

### Permutation importance

There is a second way to measure importance that does not look inside the model and so works for
any of them. Take the trained model and the test observations, measure the accuracy, and
then **shuffle** one column (its values are mixed up among the observations, breaking its
relationship with the response) and measure again:

$$
\text{imp}_{\text{perm}}(x_j) = \text{accuracy}(\text{original data}) - \text{accuracy}(\text{data with } x_j \text{ shuffled})
$$

If the accuracy drops a lot, the model depended on that variable; if it does not change, it was not really
using it. Since the shuffling is random, it is repeated several times and averaged. `scikit-learn`
provides it in `permutation_importance`.


In [ ]:

perm = permutation_importance(bosque_ruido, Xt_ruido, y_test, n_repeats=30, random_state=0, scoring="accuracy")
tabla_perm = pd.DataFrame({"impurity": imp_ruido, "permutación (caída de acierto)": perm.importances_mean, "std. dev. (30 shuffles)": perm.importances_std}, index=Xt_ruido.columns).round(3)
print(tabla_perm.sort_values("permutación (caída de acierto)", ascending=False))


By permutation, the two noise columns end up at zero (or slightly negative, which is the same thing:
shuffling them makes nothing worse). Permutation measures what matters to the business, how much worse the
prediction gets if that information were not there, and it is computed on test data, so it does not reward
variables that only served to memorize the training data.

Its weak point is the usual one with correlated variables: if you shuffle one and the model
has another almost identical, the accuracy barely drops and both look unimportant. With 73 test
firms, moreover, one percentage point is less than one firm: what is robust is which variables are
clearly above zero, not the exact order among them.


## 5. In boosting: the same, adding up round by round

In AdaBoost and in gradient boosting the trees are not averaged, they are added, but the importance is
computed with the same idea: round by round, how much each variable contributed is accumulated. For AdaBoost
with stumps, `scikit-learn` weights each stump's importance by its $\alpha$; for gradient
boosting it adds the impurity reduction of each tree over all rounds.


In [ ]:

ada = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=100, learning_rate=0.5, random_state=0).fit(X_train, y_train)
gb = GradientBoostingClassifier(n_estimators=200, max_depth=2, learning_rate=0.05, random_state=0).fit(X_train, y_train)

# AdaBoost by hand: accumulated alpha per variable (each stump uses a single variable)
alfa_por_variable = pd.Series(0.0, index=RATIOS)
for tocon, alfa in zip(ada.estimators_, ada.estimator_weights_):
    alfa_por_variable[RATIOS[tocon.tree_.feature[0]]] += alfa

tabla_boost = pd.DataFrame({"AdaBoost (accumulated alpha, normalized)": alfa_por_variable / alfa_por_variable.sum(),
                            "AdaBoost feature_importances_": ada.feature_importances_,
                            "Gradient boosting feature_importances_": gb.feature_importances_}, index=RATIOS).round(3)
print(tabla_boost)
print(f"\nTest accuracy: AdaBoost {accuracy_score(y_test, ada.predict(X_test)):.3f}   gradient boosting {accuracy_score(y_test, gb.predict(X_test)):.3f}")


The first two columns match: `scikit-learn` weights each stump's impurity-based importance by
$\alpha$, and since a stump uses a single variable, its importance is 1 for that
variable and 0 for the others; the calculation reduces to adding up the $\alpha$. XGBoost reports three versions of this: *gain* (the accumulated loss reduction, which
corresponds to the equation in section 3), *cover* (how many observations passed through the nodes
of that variable) and *weight* (in how many nodes it appeared). It is best to look at *gain*; *weight* rewards
variables with many possible splits.

A warning specific to boosting: since each tree corrects the previous ones, a variable
may appear a lot in late rounds only to fit residual noise. Impurity-based importance
counts it all the same; permutation importance on test data does not. When the two differ
a lot, it is usually a sign of overfitting.


## 6. SHAP: recovering the sign, firm by firm

None of the tree importances has a sign: they say liquidity matters, not whether a lot of
liquidity raises or lowers the risk. To recover the direction there are two paths. One is to look at the
leaves, as in Notebooks 2 and 3: the rule "sales over debt below 1.28 implies more default"
is a direction. The other, for ensembles, is the family of **per-observation contribution** methods,
of which the most widely used is SHAP (Lundberg and Lee, 2017).

The idea: each firm's prediction is split among its variables, so that

$$
f(x_i) = \phi_0 + \phi_{i1} + \phi_{i2} + \dots + \phi_{i5}
$$

where $f(x_i)$ is the model's prediction for firm $i$ (in log-odds, for a
classifier), $\phi_0$ is the average prediction, and $\phi_{ij}$ is the contribution of ratio
$j$ **for that firm**, with a sign: positive if that ratio pushed the prediction toward default,
negative if it pushed it away. The split is done with a rule from game theory (the Shapley
value) which is the only one that satisfies some reasonable fairness properties among variables.
Averaging the absolute values of $\phi_{ij}$ over all firms gives a global
importance, and looking at the sign of each $\phi_{ij}$ against the value of the ratio recovers
the direction.


In [ ]:

try:
    import shap
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "shap"], check=True)
    import shap

explicador = shap.TreeExplainer(gb)
phi = explicador.shap_values(X_test)                 # one row per test firm, one column per ratio (in log-odds)
phi = pd.DataFrame(phi, columns=RATIOS, index=X_test.index)

imp_shap = phi.abs().mean().sort_values(ascending=False)
direccion = pd.Series({r: np.corrcoef(X_test[r], phi[r])[0, 1] for r in RATIOS})
print(pd.DataFrame({"SHAP importance (mean |phi|)": imp_shap.round(3), "correlation between the ratio and its contribution": direccion[imp_shap.index].round(2),
                    "reading": np.where(direccion[imp_shap.index] > 0, "higher, more risk", "higher, less risk")}))


In [ ]:

# The classic SHAP plot: each dot is a firm; the color is the value of the ratio; the position, its contribution
shap.summary_plot(phi.values, X_test, feature_names=RATIOS, show=True)


In [ ]:

# One specific firm: why the gradient boosting gives it the probability it gives
i = X_test.index[0]
p_i = gb.predict_proba(X_test.loc[[i]])[0, 1]
print(f"Firm {i}: actual default = {y_test.loc[i]}, probability of default according to gradient boosting = {p_i:.3f}")
print(f"Average prediction (log-odds): {explicador.expected_value[0] if np.ndim(explicador.expected_value) else explicador.expected_value:.3f}")
print()
print(pd.DataFrame({"ratio value": X_test.loc[i].round(3), "training mean": X_train.mean().round(3), "contribución (log-odds)": phi.loc[i].round(3)}).sort_values("contribución (log-odds)", key=abs, ascending=False))


**On the sign of `roa`.** SHAP on the gradient boosting also finds that more
profitability pushes toward default (positive correlation between the ratio and its contribution). So
the surprise was not an artifact of the regression's linearity: it is in the data. With 146
firms it may be a coincidence of the sample, or it may be real (firms with high profitability
but high leverage, for example). The right thing to do is neither to report it nor to hide it, but to
investigate it: look at those firms before putting the model into production.

**How to read it.** For this firm, each ratio pushed the prediction in the direction and with the
strength given in the contribution column; the sum of the contributions plus the average
prediction gives the log-odds of the probability the model delivered. This is what you show
a regulator when they ask you to justify a rejection made with an ensemble: not a rule, but "these two
variables, with these values, explain most of the decision". It is the closest thing to a
coefficient that an ensemble has, with one difference: the effect of a ratio can be different
for each firm, which is precisely what the regression assumed did not happen.


## 7. All together

The table gathers, for the five ratios, the measures from this notebook, all with the same
partition.


In [ ]:

perm_bosque = permutation_importance(bosque, X_test, y_test, n_repeats=30, random_state=0, scoring="accuracy")
todo = pd.DataFrame({
    "beta logístico estandarizado": logit.params.drop("const"),
    "depth-2 tree (impurity)": arbol2.feature_importances_,
    "random forest (impurity)": bosque.feature_importances_,
    "gradient boosting (impurity)": gb.feature_importances_,
    "random forest (permutation)": perm_bosque.importances_mean,
    "SHAP on gradient boosting": imp_shap[RATIOS],
}, index=RATIOS).round(3)
print(todo)
print()
ranking = todo.copy(); ranking["beta logístico estandarizado"] = ranking["beta logístico estandarizado"].abs()
print("Ranking (1 = most important; the beta by absolute value):")
print(ranking.rank(ascending=False).astype(int))


In [ ]:

normalizado = todo.drop(columns="beta logístico estandarizado").div(todo.drop(columns="beta logístico estandarizado").sum())
fig = px.bar(normalizado.reset_index().melt(id_vars="index", var_name="medida", value_name="importancia (normalizada a 1)"),
             x="index", y="importancia (normalizada a 1)", color="medida", barmode="group", title="Five importance measures, normalized so that each sums to 1")
fig.update_layout(xaxis_title="", height=420)
fig.show()


**How to read the differences.** The numbers answer different questions. The coefficient
answers "if I move this ratio one standard deviation and leave everything else the same, how much does
the log-odds of default change, on average and assuming a linear relationship". Impurity-based importance
answers "how much did the model use this ratio to separate the training firms". Permutation
importance answers "how much accuracy do I lose on new firms if this information were not
there". SHAP answers "how much did this ratio move each firm's prediction, on average".
That a ratio is first on one list and third on another is not a contradiction, and with 146
firms all the lists are noisy.

**What it is for.** For choosing the right explanation depending on who you are talking to. To a credit
committee that demands rules: a small tree and its leaves. To a manager who asks what to look at:
forest importances, with the warning that they carry no sign. To a regulator who demands
justification for each rejection: per-client contributions. And to an econometrician who already has their
regression: the betas are still the cleanest reading when the relationship is approximately
linear, and the ensemble tells them when it is not.


## Exercises

1. **Odds ratio.** Convert the standardized beta of `razon_corriente` into an *odds ratio*
   ($e^{\beta}$) and explain it in one sentence for a manager: "one more standard deviation of current
   ratio multiplies the odds of default by ...".

2. **The regression can be permuted too.** Compute the permutation importance of the
   logistic regression (on the 73 test firms, 30 shuffles) and compare it with the standardized
   betas. Do the rankings match? Should it be surprising if they do, or if they do not?

3. **Correlated.** Add to the table a column `deuda_activos_copia` equal to
   `deuda_activos` plus a very small noise, train a random forest and look at what happens to the
   impurity-based and permutation importance of the two. Explain the result in one sentence.

4. **A regression tree.** Train a depth-2 `DecisionTreeRegressor` to predict
   `roa` from the other four ratios, and compute its importance by hand with the function
   `importancia_a_mano` (the impurity is now the variance). Compare it with the standardized
   coefficients of a linear regression of `roa` on the same four ratios.

5. **SHAP with the sign flipped.** Find in `phi` a firm for which `razon_corriente` has a
   positive contribution (pushes toward default) and another for which it has a negative contribution.
   What values of the ratio do they have? Is it consistent with the sign of the regression beta?


## Solutions


In [ ]:

# 1. Odds ratio
b = logit.params["razon_corriente"]
print(f"beta = {b:.3f}   odds ratio = exp(beta) = {np.exp(b):.3f}")
print(f"One more standard deviation of current ratio ({sd['razon_corriente']:.2f} units) multiplies the odds of default by {np.exp(b):.2f}, that is, it reduces them by {100 * (1 - np.exp(b)):.0f}%, holding the other ratios fixed.")


In [ ]:

# 2. Permuting the logistic regression
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
logit_sk = make_pipeline(StandardScaler(), LogisticRegression(C=1e6, max_iter=1000)).fit(X_train, y_train)
perm_logit = permutation_importance(logit_sk, X_test, y_test, n_repeats=30, random_state=0, scoring="accuracy")
sol2 = pd.DataFrame({"|beta estandarizado|": logit.params.drop("const").abs(), "permutación (caída de acierto)": perm_logit.importances_mean}, index=RATIOS).round(3)
sol2["beta ranking"] = sol2["|beta estandarizado|"].rank(ascending=False).astype(int); sol2["permutation ranking"] = sol2["permutación (caída de acierto)"].rank(ascending=False).astype(int)
print(sol2)
print("\nThe order is similar but not identical: the beta measures the marginal effect on the log-odds, permutation measures how much the ACCURACY drops on 73 firms,")
print("which only changes when a probability crosses the 0.5 threshold. A large beta on a variable with little dispersion in test may move few decisions.")


In [ ]:

# 3. Two almost identical variables
rng = np.random.default_rng(3)
X_dup = X_train.copy(); Xt_dup = X_test.copy()
X_dup["deuda_activos_copia"] = X_dup["deuda_activos"] + rng.normal(scale=0.001, size=len(X_dup))
Xt_dup["deuda_activos_copia"] = Xt_dup["deuda_activos"] + rng.normal(scale=0.001, size=len(Xt_dup))
bosque_dup = RandomForestClassifier(n_estimators=500, random_state=0).fit(X_dup, y_train)
perm_dup = permutation_importance(bosque_dup, Xt_dup, y_test, n_repeats=30, random_state=0, scoring="accuracy")
sol3 = pd.DataFrame({"impurity": bosque_dup.feature_importances_, "permutation": perm_dup.importances_mean}, index=X_dup.columns).round(3)
print(sol3)
print(f"\nWithout the copy, deuda_activos had impurity {bosque.feature_importances_[0]:.3f} and permutation {perm_bosque.importances_mean[0]:.3f}.")
print("With the copy, the impurity is split between the two (the forest picks one or the other at random at each split) and the permutation of each falls almost to zero:")
print("shuffling one does not hurt because the model has the other. Neither of the two measures 'knows' that they are the same information.")


In [ ]:

# 4. Regression tree for roa, against a linear regression
from sklearn.tree import DecisionTreeRegressor
SIN_ROA = [r for r in RATIOS if r != "roa"]
arbol_roa = DecisionTreeRegressor(max_depth=2, random_state=0).fit(X_train[SIN_ROA], X_train["roa"])
imp_roa = importancia_a_mano(arbol_roa.tree_, SIN_ROA)
Z4 = (X_train[SIN_ROA] - X_train[SIN_ROA].mean()) / X_train[SIN_ROA].std(ddof=0)
ols = sm.OLS(X_train["roa"], sm.add_constant(Z4)).fit()
print()
print(pd.DataFrame({"tree importance (variance)": imp_roa.round(3), "standardized beta (OLS)": ols.params.drop("const").round(4), "p value": ols.pvalues.drop("const").round(3)}))
print("\nThe tree and the regression agree on which ratio rules (ventas_deuda), and the regression adds the sign: more sales over debt, more profitability.")


In [ ]:

# 5. The same variable, contributions of opposite sign
pos = phi["razon_corriente"].idxmax(); neg = phi["razon_corriente"].idxmin()
print(pd.DataFrame({"current ratio": X_test.loc[[pos, neg], "razon_corriente"].round(3).values, "SHAP contribution": phi.loc[[pos, neg], "razon_corriente"].round(3).values, "actual default": y_test.loc[[pos, neg]].values}, index=[f"firm {pos} (pushes toward default)", f"firm {neg} (pushes away from default)"]))
print(f"\nMean current ratio in training: {X_train['razon_corriente'].mean():.3f}")
print("The firm with a positive contribution has a low current ratio; the one with a negative contribution, a high one. It is consistent with the negative beta of the")
print("regression (more liquidity, less risk), but SHAP says it firm by firm and without assuming the effect is the same over the whole range.")
